# Prepare Digitized Panel-Rows for Zenodo Upload (GM-SEUS v3.0)

GM-SEUS contains panel-row objects for arrays from either OpenStreetMap (OSM), new NAIP classification via GM-SEUS imagery classfication methods, or manual digitization from exptert interpreters. GM-SEUS v2.0 was the first to include 24,182 hand-delineated solar panel-rows (5.29 km²) composing 1,485 arrays, hosted in [Zenodo](https://doi.org/10.5281/zenodo.17042798), that did not have panel-row information in the prior GM-SEUS version. Here, we include a new round of hand-delineated panel-rows focusing on completing gaps in data for the state of Michigan. 

Note: In the original version, we attempted to upload this work to [OSM Solar Objects](https://wiki.openstreetmap.org/wiki/Tag:generator:source%3Dsolar), the result of which is documented in the Zenodo. Here, we export for public release and includsion in GM-SEUS v3.0.

The output for Zenodo here is one object: *GMSEUSv3_0_digAll.geojson*

## Import Libraries

In [15]:
# Import libraries
import pandas as pd
import geopandas as gpd
import os 
import sys

# Import packages needed for osm upload
from shapely.geometry import Polygon, MultiPolygon, LinearRing
from xml.etree.ElementTree import Element, SubElement, ElementTree
from datetime import datetime, timezone

# Import gmseusUtils
import gmseusUtils as gu

## Set Required Paths and Variables

In [16]:
# Set folder paths
wd = r'S:\Users\stidjaco\R_files\BigPanel'
downloaded_path = os.path.join(wd, r'Data\Downloaded')
derived_path = os.path.join(wd, r'Data\Derived')
derivedTemp_path = os.path.join(derived_path, r'intermediateProducts')

# Set prior version path and new panels path 
priorDigPanelsPath = os.path.join(derived_path, r'GMSEUSv3_0_newlyDigitizedPanels\GMSEUSv1_1_digAll.geojson')
newDigPanelsPath = os.path.join(derived_path, r'GMSEUSv3_0_newlyDigitizedPanels\GMSEUS2_MI_PanelRows_Merged.shp')

# Set an export folder path for GeoJSON and OSM files in prep for Zenodo
exportFolderPath = os.path.join(derived_path, r'GMSEUSv3_0_newlyDigitizedPanels')

# Set new dataset out paths: geopackage
digPanels_all_Path = os.path.join(exportFolderPath, r'GMSEUSv3_0_digAll.gpkg')
digArrays_all_Path = os.path.join(exportFolderPath, r'GMSEUSv3_0_digArraysAll.gpkg')

# Load the config from the text file
config = gu.load_config()
minPanelRowArea = config['minPanelRowArea'] # 15 m2, minimum area for a single panel row from the 1st percentile panel area from Stid et al., 2022
maxPanelRowArea = config['maxPanelRowArea'] # 254 m2 95th perccentile for a single panel row from Stid et al., 2022. MSU Solar Carport has max 1890m2
minNumPanelRows = config['minNumPanelRows'] # 3 panels, minimum number of panels rows to form a ground mounted solar array, definition from Stid et al., 2022
minPmArRatio = config['minPmArRatio'] # 18.8%, 20% was minimum ratio of panel perimeter to area ratio for panels from Stid et al., 2022, MSU Solar Carport has min 18.9%
panelArrayBuff = config['panelArrayBuff'] # 10m buffer, 20m maximum distance between panel rows to form an array. We used 5m in Stid et al., 2022, but there are lower packing factors at greater latitudes (nativeID: '1229957948')
arrayArrayBuff = config['arrayArrayBuff'] # 20m buffer, 40m maximum distance between arrays subsections of the same mount type to form a complete array. In Stid et al., 2022, we used 50m, but we checked for same installation year in addition to mount type.
overlapDist = config['overlapDist'] # 190 meters, Set a overlap distance for checking if points/mismatched geometries between Solar PV datasets are duplicates
lengthRatioThresh = config['lengthRatioThresh']  # If length ratio < 3.0, set to dual_axis or else fixed_axis_diagonal, else single- or fixed-axis
areaRatioThresh = config['areaRatioThresh']  # If area ratio < 0.15, set to fixed_diag_axis, else dual_axis
toCRS = config['to_crs']  # EPSG:6350 NAD83 (2011)
toCRS = f'EPSG:{toCRS}'

# Check and create export folder if it doesn't exist
gu.checkFolder(exportFolderPath)

## Call Panel-Rows and Determine Conflation

### Get panel-row files

In [17]:
# Call prior version and new panels and ensure target CRS (do not use gmseusUtils)
priorPanels = gpd.read_file(priorDigPanelsPath).to_crs(toCRS)
newPanels = gpd.read_file(newDigPanelsPath).to_crs(toCRS)

### Prepare prior version digitized panel-row attributes

In [18]:
# Add Source column
priorPanels['pnlSource'] = 'GMSEUSv2_0_newlyDigPanels'
priorPanels['Source'] = 'GMSEUSv2_0_newlyDigPanels'

# Add an area column and a panelID column (round to 0 decimal places)
priorPanels['area'] = priorPanels.geometry.area.round(0)
priorPanels['nativeID'] = priorPanels['panelID']

# Select only the columns needed for the final output
priorPanels = priorPanels[['nativeID', 'pnlSource', 'Source', 'area', 'geometry']]

### Prepare newly digitized panel-row attributes

In [19]:
# Add Source column
newPanels['pnlSource'] = 'GMSEUSv3_0_newlyDigPanels'
newPanels['Source'] = 'GMSEUSv3_0_newlyDigPanels'

# Add an area column and a panelID column (round to 0 decimal places)
newPanels['area'] = newPanels.geometry.area.round(0)
newPanels['nativeID'] = range(1, len(newPanels) + 1)

# Select only the columns needed for the final output
newPanels = newPanels[['nativeID', 'pnlSource', 'Source', 'area', 'geometry']]

## Drop Overlaps and Prepare for Export

In [20]:
# Save a gdf list
gdfList = [newPanels, priorPanels]

# Create merged panels using preferential spatial filtering
mergedPanels = gu.preferentialSpatialPanelRowFilter(gdfList, config)
mergedPanels = mergedPanels.reset_index(drop=True)

# Get the mount type, azimuth, length ratio, area ratio, short edge, and long edge for each panel
mergedPanels[['mount', 'azimuth', 'lengthRatio', 'shortEdge', 'longEdge']] = mergedPanels.apply(gu.assignMountType, axis=1, result_type='expand', args=(config,))
mergedPanels = mergedPanels.drop(columns=['lengthRatio', 'shortEdge', 'longEdge'])

# Create arrays from merged panels, set some attributes
mergedPanelsArrays = gu.createArrayFromPanels(mergedPanels, panelArrayBuff, '', config, 'area')
mergedPanelsArrays['Source'] = 'GMSEUSv3_0_newlyDigPanels'
mergedPanelsArrays = mergedPanelsArrays.reset_index(drop=True)

# Assign a new area column to the arrays (round to 0 decimal places)
mergedPanelsArrays['totArea'] = mergedPanelsArrays.geometry.area.round(0)

# Set an arrayID column for the arrays
mergedPanelsArrays['arrayID'] = range(1, len(mergedPanelsArrays) + 1)

# Spatially join gmseus arrays to panels, copy the arrayID to the panels, and drop the index columns. 
mergedPanels = gpd.sjoin(mergedPanels, mergedPanelsArrays[['arrayID', 'geometry']], how='left', predicate='intersects')
mergedPanels  = mergedPanels.reset_index(drop=True)
mergedPanels  = mergedPanels.drop(columns=['index_left', 'index_right'], errors='ignore') 

# Keep necessary columns for mergedPanels and mergedPanelsArrays
mergedPanels = mergedPanels[['pnlSource', 'Source', 'mount', 'azimuth', 'area', 'arrayID', 'geometry']]
mergedPanelsArrays = mergedPanelsArrays[['Source', 'arrayID', 'totArea', 'geometry']]

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ Prepare for Geopackage Export

# Export to Geopackage
mergedPanels.to_file(digPanels_all_Path, driver="GPKG")
mergedPanelsArrays.to_file(digArrays_all_Path, driver="GPKG")

# Print statistics from mergedPanels: Total panel rows, total arrays, total panel rows by source, total arrays by source, total area
print("Total panel rows in mergedPanels:", len(mergedPanels))
print("Total arrays in mergedPanelsArrays:", len(mergedPanelsArrays))
print("Total panel rows by source in mergedPanels:")
print(mergedPanels.groupby('pnlSource').size())
print("Total arrays by source in mergedPanelsArrays:")
priorArrays = gu.createArrayFromPanels(priorPanels, panelArrayBuff, '', config, 'area', False)
newArrays = gu.createArrayFromPanels(newPanels, panelArrayBuff, '', config, 'area', False)
print("Total arrays in priorPanels:", len(priorArrays))
print("Total arrays in newPanels:", len(newArrays))

# Print total area of panel rows and arrays in km2
print("Total area of panel rows in mergedPanels:", mergedPanels['area'].sum() / 1000000, "km2")
print("Total area of arrays in mergedPanelsArrays:", mergedPanelsArrays['totArea'].sum() / 1000000, "km2")

Total panel rows in mergedPanels: 31432
Total arrays in mergedPanelsArrays: 1669
Total panel rows by source in mergedPanels:
pnlSource
GMSEUSv2_0_newlyDigPanels    24182
GMSEUSv3_0_newlyDigPanels     7250
dtype: int64
Total arrays by source in mergedPanelsArrays:
Total arrays in priorPanels: 1485
Total arrays in newPanels: 184
Total area of panel rows in mergedPanels: 6.768794 km2
Total area of arrays in mergedPanelsArrays: 12.19344 km2


In [21]:
# Export as shapefiles 
mergedPanels.to_file(digPanels_all_Path.replace('.gpkg', '.shp'), driver="ESRI Shapefile")
mergedPanelsArrays.to_file(digArrays_all_Path.replace('.gpkg', '.shp'), driver="ESRI Shapefile")